In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
import mlflow

# Load and merge
txn = pd.read_csv('../data/raw/train_transaction.csv')
idn = pd.read_csv('../data/raw/train_identity.csv')
df  = txn.merge(idn, on='TransactionID', how='left')

print(f'Fraud Rate: {df["isFraud"].mean():.3%}')

# Prepare features — select numeric only for now
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['TransactionID','isFraud']]
X = df[num_cols].fillna(-999)
y = df['isFraud']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# Step 1: Anomaly score feature
iso = IsolationForest(n_estimators=200, contamination=0.035, random_state=42)
X_train['anomaly_score'] = iso.fit_predict(X_train)
X_test['anomaly_score']  = iso.predict(X_test)

# Step 2: SMOTE oversampling
smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

# Step 3: Train supervised model
with mlflow.start_run(run_name='fraud-lgbm-smote'):
    model = LGBMClassifier(
        n_estimators=400, learning_rate=0.05,
        scale_pos_weight=50, random_state=42, verbose=-1)
    model.fit(X_res, y_res)

    preds  = model.predict_proba(X_test)[:, 1]
    auc    = roc_auc_score(y_test, preds)
    mlflow.log_metric('fraud_auc', auc)
    mlflow.lightgbm.log_model(model, 'fraud-model',
                              registered_model_name='fraud-detection-lgbm')
    print(f'Fraud AUC: {auc:.4f}')
    print(classification_report(y_test, (preds > 0.5).astype(int)))


Fraud Rate: 3.499%


c:\Users\Shanu\Desktop\Final_projects\fraud_detection\credit-risk-platform\venv\lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Shanu\Desktop\Final_projects\fraud_detection\credit-risk-platform\venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "c:\Users\Shanu\Desktop\Final_projects\fraud_detection\credit-risk-platform\venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
  File "C:\Program Files\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenar

Fraud AUC: 0.9344
              precision    recall  f1-score   support

           0       0.99      0.81      0.90    113975
           1       0.15      0.89      0.25      4133

    accuracy                           0.82    118108
   macro avg       0.57      0.85      0.57    118108
weighted avg       0.97      0.82      0.87    118108

